# Random Walk → Brownian Motion

Chapter 19: Stochastic Differential Equations · *Mathematics for AI and Machine Learning* by Fuheng Wu

Compare discrete random walks at different Δt with the same seed.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-19/random-walk/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import math

import torch

torch.manual_seed(42)

# The demo's defaults: horizon T = 1, fine step 0.01
T, fine_dt = 1.0, 0.01

# ── Brownian motion as a sum of N(0, dt) steps ──────────────────────────────
n = round(T / fine_dt)
paths = (math.sqrt(fine_dt) * torch.randn(200000, n, dtype=torch.float64)).cumsum(1)
WT = paths[:, -1]
print(f"W_T over 200000 paths: mean {WT.mean().item():+.4f} (0), variance {WT.var().item():.4f} (T = {T})")
half = paths[:, n // 2 - 1]
print(f"Cov(W_0.5, W_1) = {torch.cov(torch.stack([half, WT]))[0, 1].item():.4f} (min(s, t) = 0.5)")
inc1, inc2 = half, WT - half
print(f"corr of the two half-interval increments = {torch.corrcoef(torch.stack([inc1, inc2]))[0, 1].item():+.4f} (independent)")

# ── A coin-flip walk, rescaled, looks the same (Donsker) ────────────────────
print("\n steps per unit   P(W_T > 1)   normal: P(Z > 1) =", round(0.5 * math.erfc(1 / math.sqrt(2)), 4))
for m in (4, 16, 100, 1000):
    coins = (torch.randint(0, 2, (100000, m), dtype=torch.int8).to(torch.float64) * 2 - 1)
    walk_T = coins.sum(1) / math.sqrt(m)                 # step +-1/sqrt(m) each 1/m time
    print(f"   {m:6d}          {(walk_T > 1).double().mean().item():.4f}")
# Few steps give a lumpy, discrete W_T, so the probability approaches 0.1587 only as m grows.

W_T over 200000 paths: mean -0.0006 (0), variance 1.0010 (T = 1.0)
Cov(W_0.5, W_1) = 0.5014 (min(s, t) = 0.5)
corr of the two half-interval increments = -0.0001 (independent)

 steps per unit   P(W_T > 1)   normal: P(Z > 1) = 0.1587
        4          0.0625
       16          0.1056
      100          0.1338


     1000          0.1619
